# LeetCode 1566 — Detect Pattern of Length M Repeated K or More Times bằng Python

**Link đề:** https://leetcode.com/problems/detect-pattern-of-length-m-repeated-k-or-more-times/
**Độ khó:** Easy · **Chủ đề:** Array, Enumeration

```python
class Solution:
    def containsPattern(self, arr: list[int], m: int, k: int) -> bool:
        ...
```

### Mục tiêu học
1. Hiểu "mẫu độ dài `m` lặp liên tiếp `k` lần" = một đoạn con dài `m·k` gồm `k` khối giống hệt nhau đứng sát nhau.
2. Dùng slicing `arr[i:i+m]`, phép nhân list `[1, 2] * 3`, so sánh list `==`.
3. Viết brute force đúng biên (`range(n - m*k + 1)`), đo chi phí `O(n·m·k)`.
4. Khám phá **insight dịch chuyển**: đoạn lặp ⇔ `arr[j] == arr[j + m]` cho `m·(k-1)` vị trí `j` **liên tiếp** → lời giải `O(n)` một lượt bằng bộ đếm streak.
5. Trace, chứng minh, Time/Space, test đối chiếu, bài tập biến thể (tìm mẫu, đếm số lần lặp tối đa).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — có mẫu dài m lặp lại k lần liên tiếp không?

Cho mảng số nguyên dương `arr`. Một **mẫu** (pattern) là một đoạn con (liên tiếp) gồm một hoặc nhiều giá trị, được **lặp lại liên tiếp, không chồng lấn**. Mẫu được xác định bởi độ dài `m` và số lần lặp.

Trả về `True` nếu tồn tại mẫu độ dài `m` lặp lại **ít nhất `k` lần**, ngược lại `False`.

| Ví dụ | `arr` | `m` | `k` | Output | Giải thích |
|---|---|---|---|---|---|
| 1 | `[1,2,4,4,4,4]` | 1 | 3 | `True` | `(4)` lặp 4 lần ≥ 3 |
| 2 | `[1,2,1,2,1,1,1,3]` | 2 | 2 | `True` | `(1,2)` lặp 2 lần; `(2,1)` cũng vậy |
| 3 | `[1,2,1,2,1,3]` | 2 | 3 | `False` | `(1,2)` chỉ lặp 2 lần |

**Constraints:** `2 <= arr.length <= 100`, `1 <= arr[i] <= 100`, `1 <= m <= 100`, `2 <= k <= 100`.

Lưu ý:
- "Không chồng lấn": các khối đứng **sát nhau**, khối sau bắt đầu ngay sau khối trước. Đoạn cần tìm dài đúng `m·k`.
- Nếu `m·k > n` thì chắc chắn `False`.
- Ví dụ 2: `(1,1)` ở vị trí 4–5 và 5–6 **chồng lấn** → không được tính là lặp 2 lần.

In [1]:
arr, m, k = [1, 2, 1, 2, 1, 1, 1, 3], 2, 2
for i in range(0, len(arr) - m * k + 1):
    blocks = [arr[i + t * m : i + (t + 1) * m] for t in range(k)]
    print(f"bắt đầu i={i}: các khối = {blocks} → {'LẶP' if all(b == blocks[0] for b in blocks) else 'không'}")

bắt đầu i=0: các khối = [[1, 2], [1, 2]] → LẶP
bắt đầu i=1: các khối = [[2, 1], [2, 1]] → LẶP
bắt đầu i=2: các khối = [[1, 2], [1, 1]] → không
bắt đầu i=3: các khối = [[2, 1], [1, 1]] → không
bắt đầu i=4: các khối = [[1, 1], [1, 3]] → không


Đọc output: bắt đầu tại `i = 0` (`[1,2],[1,2]`) và `i = 1` (`[2,1],[2,1]`) đều là mẫu lặp — khớp với giải thích của đề.

## 2. Nền tảng Python cần dùng

### 2.1. Slicing `arr[a:b]` và độ dài

`arr[a:b]` lấy các phần tử chỉ số `a, a+1, ..., b-1` (độ dài `b - a`). Khối thứ `t` (đếm từ 0) của mẫu bắt đầu tại `i` là `arr[i + t*m : i + (t+1)*m]`.

In [2]:
arr = [1, 2, 1, 2, 1, 3]
i, m = 0, 2
for t in range(3):
    print(f"khối t={t}: arr[{i + t*m}:{i + (t+1)*m}] = {arr[i + t*m : i + (t+1)*m]}")

khối t=0: arr[0:2] = [1, 2]
khối t=1: arr[2:4] = [1, 2]
khối t=2: arr[4:6] = [1, 3]


Khối `t = 2` là `[1, 3]` ≠ `[1, 2]` → mẫu `(1,2)` chỉ lặp 2 lần (đúng ví dụ 3). Lưu ý slicing **không báo lỗi** khi vượt biên — chỉ trả list ngắn hơn. Đó là lý do phải tự kiểm soát `i` để đoạn dài đủ `m·k`.

### 2.2. Nhân list và so sánh list

`[1, 2] * 3` → `[1, 2, 1, 2, 1, 2]` (lặp list). Hai list `==` khi cùng độ dài và từng phần tử bằng nhau.

In [3]:
print([1, 2] * 3)
print([4] * 3 == [4, 4, 4], [1, 2] * 2 == [1, 2, 1, 3])
print("slicing vượt biên:", [1, 2, 3][1:10])

[1, 2, 1, 2, 1, 2]
True False
slicing vượt biên: [2, 3]


### 2.3. Biên của vòng lặp vị trí bắt đầu

Đoạn dài `L = m·k` bắt đầu tại `i` kết thúc tại `i + L - 1 <= n - 1` → `i <= n - L` → `range(n - L + 1)`. Nếu `L > n`, `range` âm → rỗng → tự nhiên trả `False`.

In [4]:
n = 6
for m, k in [(1, 3), (2, 3), (2, 4)]:
    L = m * k
    print(f"m={m}, k={k}, L={L}: các i hợp lệ = {list(range(n - L + 1))}")

m=1, k=3, L=3: các i hợp lệ = [0, 1, 2, 3]
m=2, k=3, L=6: các i hợp lệ = [0]
m=2, k=4, L=8: các i hợp lệ = []


### 2.4. `all()` với generator

`all(cond for x in ...)` dừng ngay khi gặp phần tử sai — tiết kiệm khi so khối.

In [5]:
calls = []
def is_four(x):
    calls.append(x)
    return x == 4
print(all(is_four(x) for x in [4, 4, 1, 4, 4]), "| đã kiểm tra:", calls)

False | đã kiểm tra: [4, 4, 1]


## 3. Cách 1 — Brute force bằng slicing

Với mỗi vị trí bắt đầu `i`, lấy mẫu `p = arr[i:i+m]`, so `p * k` với đoạn `arr[i : i + m*k]`.

In [6]:
def contains_pattern_slice(arr: list[int], m: int, k: int) -> bool:
    n, L = len(arr), m * k
    for i in range(n - L + 1):
        if arr[i:i + m] * k == arr[i:i + L]:
            return True
    return False

print(contains_pattern_slice([1, 2, 4, 4, 4, 4], 1, 3))
print(contains_pattern_slice([1, 2, 1, 2, 1, 1, 1, 3], 2, 2))
print(contains_pattern_slice([1, 2, 1, 2, 1, 3], 2, 3))

True
True
False


### 3.1. Cách 2 — Ba vòng lặp (đúng gợi ý của đề), không tạo list phụ

Phần tử thứ `j` của khối thứ `t` phải bằng phần tử thứ `j` của khối đầu: `arr[i + t*m + j] == arr[i + j]`.

In [7]:
def contains_pattern_loops(arr: list[int], m: int, k: int) -> bool:
    n = len(arr)
    for i in range(n - m * k + 1):
        ok = True
        for t in range(1, k):                  # khối 1..k-1 so với khối 0
            for j in range(m):
                if arr[i + t * m + j] != arr[i + j]:
                    ok = False
                    break
            if not ok:
                break
        if ok:
            return True
    return False

print(contains_pattern_loops([1, 2, 4, 4, 4, 4], 1, 3))
print(contains_pattern_loops([1, 2, 1, 2, 1, 1, 1, 3], 2, 2))
print(contains_pattern_loops([1, 2, 1, 2, 1, 3], 2, 3))

True
True
False


Cờ `ok` + hai lệnh `break` để thoát **hai tầng** vòng lặp khi phát hiện sai khác — Python không có `break 2`.

### 3.2. Chi phí brute force

Có `n - m·k + 1` vị trí, mỗi vị trí so tối đa `m·(k-1)` cặp → `O(n·m·k)`; với `n = 100` thì `m·k <= 100` nên tệ nhất cỡ `(n - L)·L ≈ 2 500` phép — rất nhỏ. Nhưng hãy xem **công việc bị lặp lại**: vị trí `i` và `i + 1` so sánh gần như **cùng những cặp phần tử**!

In [8]:
def compared_pairs(arr, m, k, i):
    return {(i + t * m + j, i + j) for t in range(1, k) for j in range(m)}

arr, m, k = [1, 2, 1, 2, 1, 2, 1, 3], 2, 3
p0, p1 = compared_pairs(arr, m, k, 0), compared_pairs(arr, m, k, 1)
print("i=0 so:", sorted(p0))
print("i=1 so:", sorted(p1))
print("Dạng chung: mọi cặp đều là (j + c·m, j) — khoảng cách là bội của m")

i=0 so: [(2, 0), (3, 1), (4, 0), (5, 1)]
i=1 so: [(3, 1), (4, 2), (5, 1), (6, 2)]
Dạng chung: mọi cặp đều là (j + c·m, j) — khoảng cách là bội của m


## 4. Insight — so mỗi phần tử với phần tử cách nó đúng `m` bước

Nếu đoạn `arr[i : i + m·k]` là `k` khối giống nhau, thì **mọi** phần tử trong đoạn (trừ khối cuối) bằng phần tử **cách nó `m` vị trí** về bên phải:

```text
arr[j] == arr[j + m]   với mọi j = i, i+1, ..., i + m·(k-1) - 1
```

Đây là `m·(k-1)` vị trí `j` **liên tiếp**. Và điều ngược lại cũng đúng: nếu `arr[j] == arr[j + m]` với `m·(k-1)` giá trị `j` liên tiếp bắt đầu từ `i`, thì nối chuỗi các đẳng thức ta có `arr[i + t·m + r] == arr[i + r]` → `k` khối giống nhau.

Vậy bài toán trở thành: **có chuỗi liên tiếp dài ≥ `m·(k-1)` các vị trí `j` thỏa `arr[j] == arr[j+m]` không?** — chính là bài "streak" (giống *Three Consecutive Odds*)!

In [9]:
arr, m = [1, 2, 1, 2, 1, 1, 1, 3], 2
match = [arr[j] == arr[j + m] for j in range(len(arr) - m)]
print("arr       :", arr)
print("j         :", list(range(len(arr) - m)))
print("arr[j]==arr[j+m]:", [int(x) for x in match])
print("Cần chuỗi 1 liên tiếp dài m*(k-1) =", m * (2 - 1), "với k=2")

arr       : [1, 2, 1, 2, 1, 1, 1, 3]
j         : [0, 1, 2, 3, 4, 5]
arr[j]==arr[j+m]: [1, 1, 1, 0, 1, 0]
Cần chuỗi 1 liên tiếp dài m*(k-1) = 2 với k=2


Chuỗi `1,1` ở `j = 0, 1` (dài 2 = `m·(k-1)`) → có mẫu lặp 2 lần bắt đầu ở `j = 0` ✓.

## 5. Lời giải chính — Một lượt với bộ đếm streak

### 5.1. Từng đoạn
1. `need = m * (k - 1)`; `streak = 0`.
2. Duyệt `j` từ `0` đến `n - m - 1`:
   - Nếu `arr[j] == arr[j + m]` → `streak += 1`; nếu `streak == need` → `return True`.
   - Ngược lại → `streak = 0`.
3. Hết vòng → `return False`.

### 5.2. Ghép lại

In [10]:
def contains_pattern_shift(arr: list[int], m: int, k: int) -> bool:
    need = m * (k - 1)
    streak = 0
    for j in range(len(arr) - m):
        if arr[j] == arr[j + m]:
            streak += 1
            if streak == need:
                return True
        else:
            streak = 0
    return False

print(contains_pattern_shift([1, 2, 4, 4, 4, 4], 1, 3))
print(contains_pattern_shift([1, 2, 1, 2, 1, 1, 1, 3], 2, 2))
print(contains_pattern_shift([1, 2, 1, 2, 1, 3], 2, 3))

True
True
False


Nếu `m >= n`, `range(n - m)` rỗng → `False` — đúng, vì không thể lặp ≥ 2 lần.

## 6. TRACE từng bước

Ví dụ 3: `arr = [1,2,1,2,1,3]`, `m = 2`, `k = 3` → `need = 4`.

| j | arr[j] | arr[j+2] | bằng? | streak |
|---|---|---|---|---|
| 0 | 1 | 1 | ✓ | 1 |
| 1 | 2 | 2 | ✓ | 2 |
| 2 | 1 | 1 | ✓ | 3 |
| 3 | 2 | 3 | ✗ | 0 |

Không lúc nào đạt 4 → `False`. Nếu `arr[5]` là `2` thì `j = 3` khớp → streak 4 → `True` (mẫu `1,2` lặp 3 lần).

In [11]:
def trace_shift(arr, m, k):
    need = m * (k - 1); streak = 0
    print(f"arr={arr}, m={m}, k={k}, need = m*(k-1) = {need}")
    for j in range(len(arr) - m):
        eq = arr[j] == arr[j + m]
        streak = streak + 1 if eq else 0
        print(f"j={j}: arr[{j}]={arr[j]} vs arr[{j+m}]={arr[j+m]} → {'=' if eq else '≠'}, streak={streak}")
        if streak == need:
            print(f"→ đạt {need}: mẫu arr[{j - need + 1}:{j - need + 1 + m}] = {arr[j - need + 1 : j - need + 1 + m]} lặp {k} lần")
            return True
    print("→ False")
    return False

assert trace_shift([1, 2, 1, 2, 1, 3], 2, 3) is False
print()
assert trace_shift([1, 2, 1, 2, 1, 2], 2, 3) is True
print()
assert trace_shift([1, 2, 4, 4, 4, 4], 1, 3) is True

arr=[1, 2, 1, 2, 1, 3], m=2, k=3, need = m*(k-1) = 4
j=0: arr[0]=1 vs arr[2]=1 → =, streak=1
j=1: arr[1]=2 vs arr[3]=2 → =, streak=2
j=2: arr[2]=1 vs arr[4]=1 → =, streak=3
j=3: arr[3]=2 vs arr[5]=3 → ≠, streak=0
→ False

arr=[1, 2, 1, 2, 1, 2], m=2, k=3, need = m*(k-1) = 4
j=0: arr[0]=1 vs arr[2]=1 → =, streak=1
j=1: arr[1]=2 vs arr[3]=2 → =, streak=2
j=2: arr[2]=1 vs arr[4]=1 → =, streak=3
j=3: arr[3]=2 vs arr[5]=2 → =, streak=4
→ đạt 4: mẫu arr[0:2] = [1, 2] lặp 3 lần

arr=[1, 2, 4, 4, 4, 4], m=1, k=3, need = m*(k-1) = 2
j=0: arr[0]=1 vs arr[1]=2 → ≠, streak=0
j=1: arr[1]=2 vs arr[2]=4 → ≠, streak=0
j=2: arr[2]=4 vs arr[3]=4 → =, streak=1
j=3: arr[3]=4 vs arr[4]=4 → =, streak=2
→ đạt 2: mẫu arr[2:3] = [4] lặp 3 lần


Vị trí bắt đầu mẫu khi streak đạt `need` tại `j` là `j - need + 1` (đầu chuỗi streak).

## 7. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Slicing | `arr[i:i+m]*k == arr[i:i+mk]` | `O(n·m·k)` | `O(m·k)` | ngắn, dễ đọc |
| 2. Ba vòng lặp | so từng phần tử với khối đầu | `O(n·m·k)` | `O(1)` | đúng gợi ý đề |
| 3. **Dịch `m` + streak** | `arr[j]==arr[j+m]` liên tiếp `m(k-1)` lần | **`O(n)`** | `O(1)` | **tối ưu**, một lượt |

## 8. Code nộp LeetCode

In [12]:
class Solution:
    def containsPattern(self, arr: list[int], m: int, k: int) -> bool:
        need = m * (k - 1)
        streak = 0
        for j in range(len(arr) - m):
            if arr[j] == arr[j + m]:
                streak += 1
                if streak == need:
                    return True
            else:
                streak = 0
        return False

sol = Solution()
print(sol.containsPattern([1, 2, 4, 4, 4, 4], 1, 3))
print(sol.containsPattern([1, 2, 1, 2, 1, 1, 1, 3], 2, 2))
print(sol.containsPattern([1, 2, 1, 2, 1, 3], 2, 3))

True
True
False


Phiên bản brute force slicing (cũng Accepted, dễ nhớ):

```python
class Solution:
    def containsPattern(self, arr: list[int], m: int, k: int) -> bool:
        for i in range(len(arr) - m * k + 1):
            if arr[i:i + m] * k == arr[i:i + m * k]:
                return True
        return False
```

## 9. Vì sao đúng?

**Mệnh đề:** đoạn `arr[i : i + m·k]` là `k` khối giống nhau ⇔ `arr[j] == arr[j + m]` với mọi `j ∈ [i, i + m(k-1) - 1]`.

- (⇒) Phần tử `j = i + t·m + r` (với `0 <= t < k-1`, `0 <= r < m`) nằm ở khối `t`; `j + m` nằm cùng vị trí `r` ở khối `t + 1`. Hai khối giống nhau → bằng nhau ✓.
- (⇐) Với mỗi `r` cố định, chuỗi `arr[i+r] = arr[i+r+m] = arr[i+r+2m] = ... = arr[i+r+(k-1)m]` (mỗi dấu `=` là một vị trí `j` trong khoảng) → phần tử thứ `r` của mọi khối bằng nhau → `k` khối giống nhau ✓.

**Streak:** sau bước `j`, `streak` = số vị trí liên tiếp **kết thúc tại `j`** thỏa `arr[j] == arr[j+m]`. Tồn tại đoạn lặp ⇔ tồn tại khoảng liên tiếp dài `need` các vị trí thỏa ⇔ `streak` đạt `need` tại cuối khoảng đó. Vì `streak` tăng từng đơn vị, nó đi qua đúng giá trị `need` → không bỏ sót.

**Biên:** khoảng `j ∈ [i, i + need - 1]` cần `j + m <= n - 1`, tức `i + m·k - 1 <= n - 1` — đúng điều kiện đoạn nằm trong mảng. Vòng `range(n - m)` đảm bảo `j + m` hợp lệ.

## 10. Time / Space

Gọi `n = len(arr)`.

- **Lời giải chính:** một vòng `n - m` bước, mỗi bước `O(1)` → **Time `O(n)`**, **Space `O(1)`**. Không phụ thuộc `m`, `k`.
- **Brute force:** `(n - mk + 1)` vị trí × `O(mk)` → `O(n·m·k)`; slicing tốn thêm `O(mk)` bộ nhớ tạm.

Với `n ≤ 100`, cả hai đều tức thời; cách dịch `m` là kỹ thuật đáng học vì dùng lại ở nhiều bài chuỗi/mảng tuần hoàn.

## 11. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.containsPattern([1, 2, 4, 4, 4, 4], 1, 3), True)
check("đề 2", sol.containsPattern([1, 2, 1, 2, 1, 1, 1, 3], 2, 2), True)
check("đề 3", sol.containsPattern([1, 2, 1, 2, 1, 3], 2, 3), False)
check("m*k > n", sol.containsPattern([1, 1], 1, 3), False)
check("m*k == n", sol.containsPattern([2, 2], 1, 2), True)
check("m >= n", sol.containsPattern([1, 2], 2, 2), False)
check("chồng lấn không tính", sol.containsPattern([1, 1, 1], 2, 2), False)
check("ở cuối mảng", sol.containsPattern([5, 1, 2, 3, 1, 2, 3], 3, 2), True)
check("gần đủ rồi gãy", sol.containsPattern([2, 2, 1, 2, 2, 1, 2, 2, 3], 3, 3), False)
check("toàn bằng nhau", sol.containsPattern([7] * 100, 10, 10), True)

PASS đề 1: True
PASS đề 2: True
PASS đề 3: False
PASS m*k > n: False
PASS m*k == n: True
PASS m >= n: False
PASS chồng lấn không tính: False
PASS ở cuối mảng: True
PASS gần đủ rồi gãy: False
PASS toàn bằng nhau: True


`[1,1,1]`, `m = 2`, `k = 2`: mẫu `(1,1)` cần đoạn dài 4 — mảng chỉ dài 3 → `False` (hai bản `(1,1)` chỉ có được nếu chồng lấn).

**Phản ví dụ cho cách "đếm số lần xuất hiện của mẫu":** đếm `p` xuất hiện ≥ `k` lần **ở bất kỳ đâu** là sai — đề yêu cầu **liên tiếp**.

In [14]:
def contains_pattern_anywhere_wrong(arr, m, k):
    n = len(arr)
    for i in range(n - m + 1):
        p = arr[i:i + m]
        occ = sum(1 for j in range(n - m + 1) if arr[j:j + m] == p)
        if occ >= k:
            return True
    return False

arr = [1, 2, 9, 1, 2, 9, 1, 2]
print("đếm rải rác:", contains_pattern_anywhere_wrong(arr, 2, 3), "| đúng:", sol.containsPattern(arr, 2, 3))

đếm rải rác: True | đúng: False


### 11.1. Đối chiếu 3 cách trên input ngẫu nhiên

Dùng bảng giá trị nhỏ (`1..3`) để mẫu lặp xuất hiện thường xuyên.

In [15]:
import random
rng = random.Random(1566)
true_count = 0
for _ in range(5000):
    n = rng.randint(2, 16)
    arr = [rng.randint(1, rng.choice([1, 2, 3])) for _ in range(n)]
    m = rng.randint(1, 5); k = rng.randint(2, 5)
    exp = contains_pattern_slice(arr, m, k)
    true_count += exp
    assert contains_pattern_loops(arr, m, k) == exp
    assert contains_pattern_shift(arr, m, k) == exp
    assert Solution().containsPattern(arr, m, k) == exp
print(f"5000 test ngẫu nhiên: 3 cách khớp nhau ({true_count} ca True, {5000 - true_count} ca False)")

5000 test ngẫu nhiên: 3 cách khớp nhau (815 ca True, 4185 ca False)


In thêm số ca `True/False` để chắc rằng bộ test **không lệch** về một phía (nếu toàn `False` thì test chẳng kiểm tra được gì).

## 12. Lỗi người mới hay gặp

### Lỗi 1 — Biên vòng lặp `range(n - m*k)` (thiếu `+ 1`)
Bỏ sót đoạn kết thúc đúng ở phần tử cuối.

In [16]:
def slice_off_by_one_bug(arr, m, k):
    for i in range(len(arr) - m * k):          # SAI: thiếu + 1
        if arr[i:i + m] * k == arr[i:i + m * k]:
            return True
    return False

arr = [5, 1, 2, 3, 1, 2, 3]
print("lỗi:", slice_off_by_one_bug(arr, 3, 2), "| đúng:", contains_pattern_slice(arr, 3, 2))
print("lỗi:", slice_off_by_one_bug([2, 2], 1, 2), "| đúng:", contains_pattern_slice([2, 2], 1, 2))

lỗi: False | đúng: True
lỗi: False | đúng: True


### Lỗi 2 — Không kiểm soát biên khi dùng slicing
Nếu chạy `i` tới `n - m` mà không đảm bảo đoạn đủ dài `m·k`, slicing **không báo lỗi** mà trả list ngắn → so `p * k` (dài `mk`) với list ngắn luôn `False`, nên kết quả vẫn đúng **nhờ may mắn**. Nhưng với cách ba vòng lặp truy cập chỉ số trực tiếp thì sẽ `IndexError`. Luôn suy nghĩ biên rõ ràng: `i <= n - m·k`.

### Lỗi 3 — Streak cần `m·k` thay vì `m·(k-1)`
Số so sánh dịch `m` chỉ có `m·(k-1)` (khối cuối không có "khối sau" để so).

In [17]:
def shift_need_bug(arr, m, k):
    need = m * k                               # SAI
    streak = 0
    for j in range(len(arr) - m):
        streak = streak + 1 if arr[j] == arr[j + m] else 0
        if streak == need:
            return True
    return False

print("lỗi:", shift_need_bug([1, 2, 1, 2], 2, 2), "| đúng:", contains_pattern_shift([1, 2, 1, 2], 2, 2))

lỗi: False | đúng: True


### Lỗi 4 — Quên reset `streak` khi gặp vị trí không khớp
Khi đó các vị trí khớp **rải rác** được cộng dồn → `True` sai.

In [18]:
def shift_no_reset_bug(arr, m, k):
    need = m * (k - 1); streak = 0
    for j in range(len(arr) - m):
        if arr[j] == arr[j + m]:
            streak += 1
            if streak == need:
                return True
    return False

arr = [1, 2, 1, 3, 1, 3, 1, 5, 1]
print("so khớp:", [int(arr[j] == arr[j + 2]) for j in range(len(arr) - 2)])
print("lỗi:", shift_no_reset_bug(arr, 2, 3), "| đúng:", contains_pattern_shift(arr, 2, 3))

so khớp: [1, 0, 1, 1, 1, 0, 1]
lỗi: True | đúng: False


### Lỗi 5 — Hiểu "k or more" thành "đúng k"
Mẫu lặp 4 lần cũng thỏa `k = 3` (ví dụ 1). Không được kiểm tra "khối thứ `k+1` phải khác".

## 13. Mẹo nhận dạng pattern

- "Đoạn gồm các khối lặp lại có chu kỳ `m`" ⇔ "**`arr[j] == arr[j+m]`** trên một khoảng" — mẹo **so với chính mình dịch đi một chu kỳ**. Gặp lại ở 459 *Repeated Substring Pattern*, 686 *Repeated String Match*, thuật toán KMP/Z-function.
- "Có khoảng **liên tiếp** dài ≥ L thỏa điều kiện P" → **streak counter**.
- Constraints nhỏ (`n ≤ 100`) → brute force slicing là lựa chọn an toàn khi đi thi; biết cách `O(n)` để gây ấn tượng khi phỏng vấn.
- Đề có chữ "consecutive / without overlapping" → các khối **sát nhau**, độ dài đoạn = `m·k`.

## 14. Biến thể

### 14.1. Trả về vị trí bắt đầu và mẫu

In [19]:
def find_pattern(arr, m, k):
    need = m * (k - 1); streak = 0
    for j in range(len(arr) - m):
        streak = streak + 1 if arr[j] == arr[j + m] else 0
        if streak == need:
            start = j - need + 1
            return start, arr[start:start + m]
    return -1, None

print(find_pattern([1, 2, 1, 2, 1, 1, 1, 3], 2, 2))
print(find_pattern([3, 1, 4, 4, 4, 4], 1, 3))
print(find_pattern([1, 2, 1, 2, 1, 3], 2, 3))

(0, [1, 2])
(2, [4])
(-1, None)


### 14.2. Số lần lặp **tối đa** của một mẫu độ dài `m`
Streak dài nhất `S` (của `arr[j]==arr[j+m]`) cho số lần lặp `S // m + 1`.

In [20]:
def max_repeats(arr, m):
    best = streak = 0
    for j in range(len(arr) - m):
        streak = streak + 1 if arr[j] == arr[j + m] else 0
        best = max(best, streak)
    return best // m + 1 if len(arr) >= m else 0

print(max_repeats([1, 2, 4, 4, 4, 4], 1))      # 4
print(max_repeats([1, 2, 1, 2, 1, 3], 2))      # 2
print(max_repeats([1, 2, 3], 2))               # 1 (chỉ có một bản)

4
2
1


### 14.3. Chuỗi: có phải lặp của một chuỗi con? (LeetCode 459)
Mẹo nổi tiếng: `s` là lặp của chuỗi con ⇔ `s in (s + s)[1:-1]`.

In [21]:
def repeated_substring_pattern(s):
    return s in (s + s)[1:-1]

for s in ["abab", "aba", "abcabcabc", "a"]:
    print(s, repeated_substring_pattern(s))

abab True
aba False
abcabcabc True
a False


## 15. Bài tập (TODO không tự chạy)

### Bài 1 — Tính tay
`arr = [2, 1, 2, 1, 2, 1, 5]`, `m = 2`, `k = 3`. Lập bảng `j, arr[j], arr[j+2], streak`. Kết quả?

### Bài 2 — Đoán output
```python
arr = [4, 4, 4, 4]
print(arr[0:1] * 3 == arr[0:3], [arr[j] == arr[j + 1] for j in range(3)])
```

### Bài 3 — TODO: brute force slicing

In [22]:
def exercise_contains_pattern_slice(arr, m, k):
    # TODO: với i trong range(len(arr) - m*k + 1), so arr[i:i+m]*k với arr[i:i+m*k]
    pass

# assert exercise_contains_pattern_slice([1, 2, 4, 4, 4, 4], 1, 3) is True
# assert exercise_contains_pattern_slice([1, 2, 1, 2, 1, 3], 2, 3) is False

### Bài 4 — TODO: lời giải `O(n)` dịch `m`

In [23]:
def exercise_contains_pattern_shift(arr, m, k):
    # TODO 1: need = m * (k - 1)
    # TODO 2: streak của arr[j] == arr[j + m], reset khi khác
    # TODO 3: return True khi streak == need
    pass

# assert exercise_contains_pattern_shift([1, 2, 1, 2, 1, 1, 1, 3], 2, 2) is True
# assert exercise_contains_pattern_shift([1, 2, 1, 2, 1, 3], 2, 3) is False

### Bài 5 — TODO: độ dài `m` nhỏ nhất
Tìm `m` **nhỏ nhất** (`1 <= m`) sao cho có mẫu độ dài `m` lặp ít nhất `k` lần; không có thì `-1`.

In [24]:
def exercise_min_m(arr, k):
    # TODO: thử m = 1, 2, ..., len(arr) // k với hàm O(n) ở bài 4
    pass

# assert exercise_min_m([1, 2, 1, 2, 1, 2], 3) == 2
# assert exercise_min_m([1, 2, 4, 4, 4, 4], 3) == 1
# assert exercise_min_m([1, 2, 3], 2) == -1

### Bài 6 — Mini challenge: chuỗi tuần hoàn
Cho chuỗi `s`, trả về chu kỳ nhỏ nhất `p` (`1 <= p <= len(s)`) sao cho `s[i] == s[i + p]` với **mọi** `i` hợp lệ (chu kỳ không cần chia hết độ dài, ví dụ `"abcab"` có chu kỳ 3).

In [25]:
def exercise_smallest_period(s: str) -> int:
    # TODO: thử p từ 1 tới len(s); kiểm tra all(s[i] == s[i + p] for i in range(len(s) - p))
    pass

# assert exercise_smallest_period("abcab") == 3
# assert exercise_smallest_period("aaaa") == 1
# assert exercise_smallest_period("abcd") == 4

## 16. Gợi ý 3 mức

**Bài 1:** (1) `need = 2·2 = 4`. (2) So `arr[j]` với `arr[j+2]` cho `j = 0..4`. (3) Đếm streak, xem có đạt 4.

**Bài 2:** (1) Nhân list. (2) So từng cặp kề. (3) Một `bool` và một list 3 phần tử.

**Bài 3:** (1) Biên `n - m*k + 1`. (2) `arr[i:i+m] * k`. (3) Xem mục 3.

**Bài 4:** (1) Như *Three Consecutive Odds* nhưng điều kiện là `arr[j] == arr[j+m]`. (2) `range(len(arr) - m)`. (3) Xem mục 5.

**Bài 5:** (1) `m·k <= n` nên `m <= n // k`. (2) Thử `m` tăng dần, gặp `True` là dừng. (3) Tổng `O(n²/k)`.

**Bài 6:** (1) `p = len(s)` luôn thỏa (không có `i` nào để kiểm). (2) Thử `p` tăng dần. (3) `all(...)` trên `range(len(s) - p)`.

## 17. Đáp án tham khảo

**Bài 1:** `need = 4`.

| j | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| arr[j] vs arr[j+2] | 2=2 | 1=1 | 2=2 | 1=1 | 2≠5 |
| streak | 1 | 2 | 3 | 4 → `True` | |

Mẫu `(2,1)` lặp 3 lần ở vị trí 0..5.

**Bài 2:** `True [True, True, True]`.

In [26]:
def answer_contains_pattern_slice(arr, m, k):
    for i in range(len(arr) - m * k + 1):
        if arr[i:i + m] * k == arr[i:i + m * k]:
            return True
    return False

def answer_contains_pattern_shift(arr, m, k):
    need = m * (k - 1); streak = 0
    for j in range(len(arr) - m):
        if arr[j] == arr[j + m]:
            streak += 1
            if streak == need:
                return True
        else:
            streak = 0
    return False

def answer_min_m(arr, k):
    for m in range(1, len(arr) // k + 1):
        if answer_contains_pattern_shift(arr, m, k):
            return m
    return -1

def answer_smallest_period(s):
    for p in range(1, len(s) + 1):
        if all(s[i] == s[i + p] for i in range(len(s) - p)):
            return p
    return len(s)

assert answer_contains_pattern_shift([2, 1, 2, 1, 2, 1, 5], 2, 3) is True
assert answer_contains_pattern_slice([1, 2, 4, 4, 4, 4], 1, 3) is True
assert answer_contains_pattern_slice([1, 2, 1, 2, 1, 3], 2, 3) is False
assert answer_contains_pattern_shift([1, 2, 1, 2, 1, 1, 1, 3], 2, 2) is True
assert answer_contains_pattern_shift([1, 2, 1, 2, 1, 3], 2, 3) is False
assert answer_min_m([1, 2, 1, 2, 1, 2], 3) == 2
assert answer_min_m([1, 2, 4, 4, 4, 4], 3) == 1
assert answer_min_m([1, 2, 3], 2) == -1
assert answer_smallest_period("abcab") == 3
assert answer_smallest_period("aaaa") == 1
assert answer_smallest_period("abcd") == 4
arr = [4, 4, 4, 4]
assert (arr[0:1] * 3 == arr[0:3], [arr[j] == arr[j + 1] for j in range(3)]) == (True, [True, True, True])
rng = random.Random(66)
for _ in range(500):
    a = [rng.randint(1, 2) for _ in range(rng.randint(2, 12))]
    kk = rng.randint(2, 4)
    brute = next((m for m in range(1, len(a) // kk + 1) if contains_pattern_slice(a, m, kk)), -1)
    assert answer_min_m(a, kk) == brute
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


`next(generator, mặc_định)` lấy phần tử **đầu tiên** của generator, hoặc trả `mặc_định` nếu generator rỗng — cách gọn để viết "tìm phần tử đầu tiên thỏa điều kiện".

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def containsPattern(self, arr: list[int], m: int, k: int) -> bool:
        need, streak = m * (k - 1), 0
        for j in range(len(arr) - m):
            if arr[j] == arr[j + m]:         # so với chính mình dịch m
                streak += 1
                if streak == need:
                    return True
            else:
                streak = 0
        return False
```

**Câu thần chú:** *"k khối chu kỳ m ⇔ arr[j] == arr[j+m] liên tiếp m·(k−1) lần."*

**Checklist trước khi submit**
- [ ] `need = m * (k - 1)`, không phải `m * k`.
- [ ] Reset `streak = 0` khi không khớp.
- [ ] `range(len(arr) - m)` để `arr[j + m]` không vượt biên.
- [ ] Nếu brute force: `range(len(arr) - m*k + 1)` (có `+ 1`).
- [ ] Test: `m*k == n`, `m*k > n`, mẫu ở cuối mảng, mẫu gần đủ rồi gãy.
- [ ] Đúng tên method `containsPattern`.